# PageForge v2 — 节点流程模拟

不启动服务，直接调用各节点函数，模拟完整 graph 流程。

**测试场景**：`"帮我做一个极简风格的 Todo 应用"` (code_gen → 完整6节点流程)

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
os.chdir('/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend')
print(f'cwd: {os.getcwd()}')
print(f'Python: {sys.version}')

cwd: /Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend
Python: 3.13.3 (main, Apr  8 2025, 13:54:08) [Clang 17.0.0 (clang-1700.0.13.3)]


## Step 0: 注入 mock 事件发射器

不启动 SSE 服务，用全局 `set_event_emitter` 把事件打印到控制台。

In [2]:
from app.graph.nodes.event_emitter import set_event_emitter

# 全局事件日志
event_log = []

def mock_emitter(event_type: str, data: dict):
    """把 SSE 事件打印出来"""
    import time
    ts = time.strftime('%H:%M:%S')
    entry = {'ts': ts, 'type': event_type, 'data': data}
    event_log.append(entry)
    # 简化打印
    short_data = {k: (v[:60] + '...' if isinstance(v, str) and len(v) > 60 else v) 
                  for k, v in data.items()}
    print(f'  📡 [{ts}] {event_type}  {short_data}')

set_event_emitter(mock_emitter)
print('✅ 事件发射器已注入')

✅ 事件发射器已注入


## Step 1: 构造初始 State

模拟用户输入 → 初始 AgentState

In [3]:
import uuid

user_message = "帮我做一个极简风格的 Todo 应用，支持添加、删除、标记完成"
session_id = str(uuid.uuid4())[:8]

initial_state = {
    'user_message': user_message,
    'session_id': session_id,
    'base_html': '',
    'task_list': [],
    'current_html': '',
    'validation_errors': [],
    'iteration_count': 0,
    'fix_count': 0,
    'response_message': '',
    'output_html': None,
    'output_version': 0,
    'is_complete': False,
    'project_type': None,
    'files': [],
    'project_id': None,
    'install_status': '',
    'dev_server_url': None,
    'ui_style': None,
    'intent': None,
    'confidence': 0.0,
    'tags': [],
    'mode': None,
    'complexity': None,
    'model_strategy': {},
    'thought_summary': '',
    'plan_steps': [],
    'ui_style_config': '',
    'status': '',
}

print(f'用户消息: {user_message}')
print(f'Session: {session_id}')
print(f'State keys: {list(initial_state.keys())}')
print('\n--- 开始模拟 ---')

用户消息: 帮我做一个极简风格的 Todo 应用，支持添加、删除、标记完成
Session: ff5a8e87
State keys: ['user_message', 'session_id', 'base_html', 'task_list', 'current_html', 'validation_errors', 'iteration_count', 'fix_count', 'response_message', 'output_html', 'output_version', 'is_complete', 'project_type', 'files', 'project_id', 'install_status', 'dev_server_url', 'ui_style', 'intent', 'confidence', 'tags', 'mode', 'complexity', 'model_strategy', 'thought_summary', 'plan_steps', 'ui_style_config', 'status']

--- 开始模拟 ---


## Step 2: intent_router — 意图识别

第一个节点，分析用户输入 → intent/model_strategy

In [4]:
print('=' * 60)
print('🔍 Node 1: intent_router')
print('=' * 60)

from app.graph.nodes.intent_router import intent_router

state_after_intent = intent_router(initial_state)

print(f'\n📊 结果:')
print(f'   intent      = {state_after_intent.get("intent")}')
print(f'   confidence  = {state_after_intent.get("confidence")}')
print(f'   tags        = {state_after_intent.get("tags")}')
print(f'   mode        = {state_after_intent.get("mode")}')
print(f'   complexity  = {state_after_intent.get("complexity")}')
print(f'   ui_style    = {state_after_intent.get("ui_style")}')
print(f'   model_strategy = {state_after_intent.get("model_strategy")}')

🔍 Node 1: intent_router

📊 结果:
   intent      = code_gen
   confidence  = 0.95
   tags        = ['todo', 'frontend', 'minimal']
   mode        = frontend
   complexity  = simple
   ui_style    = minimal
   model_strategy = {'thinking': 'chat', 'plan': 'chat', 'style_picker': 'chat', 'code_gen': 'pro', 'reply': 'chat'}


## Step 3: thinking — 思维链

LLM 思考过程 → thought_summary

In [5]:
print('\n' + '=' * 60)
print('🧠 Node 2: thinking')
print('=' * 60)

from app.graph.nodes.thinking import thinking_node

state_after_thinking = thinking_node(state_after_intent)

print(f'\n📊 结果:')
print(f'   thought_summary = {state_after_thinking.get("thought_summary", "")[:120]}...')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "thinking" in e["type"]])}')


🧠 Node 2: thinking
  📡 [14:26:32] thinking:start  {'id': 'thinking_1777962392234', 'timestamp': 1777962392.235397}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '1', 'timestamp': 1777962398.836864}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '.', 'timestamp': 1777962398.837996}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': ' 需求', 'timestamp': 1777962398.838449}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '理解：', 'timestamp': 1777962398.838777}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '  \n', 'timestamp': 1777962398.839016}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '用户需要一个', 'timestamp': 1777962398.839245}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '极简', 'timestamp': 1777962398.839523}
  📡 [14:26:38] thinking:delta  {'id': 'thinking_1777962392234', 'delta': '风格的 Todo', 'timestamp': 177

## Step 4: plan — 计划制定

LLM 生成执行计划 → plan_steps

In [7]:
print('\n' + '=' * 60)
print('📋 Node 3: plan')
print('=' * 60)

from app.graph.nodes.plan import plan_node

state_after_plan = plan_node(state_after_thinking)

print(f'\n📊 结果:')
steps = state_after_plan.get('plan_steps', [])
print(f'   共 {len(steps)} 个步骤:')
for s in steps:
        print(f'   Step {s["id"]}: {s["label"]} (type={s.get("type", "?")})')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "plan" in e["type"]])}')


📋 Node 3: plan
  📡 [14:29:40] plan:start  {'id': 'plan_1777962580590', 'steps': [], 'current': 0, 'timestamp': 1777962580.593558}


[Plan] LLM 输出 JSON 解析失败: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Expecting value: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 2 (char 1)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Expecting value: line 1 column 1 (char 0)，使用降级方案
[stream_llm] LLM 调用失败 node=plan: 'int' object has no attribute 'get'
Traceback (most recent call last):
  File "/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend/app/graph/nodes/llm_utils.py", line 58, in stream_llm
    emit_delta(text)
    ~~~~~~~~~~^^^^^^
  File "/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend/app/graph/nodes/plan.py", line 125, in <lambda>
    steps=_parse_plan_steps(chunk, int

  📡 [14:29:43] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962583.768387}
  📡 [14:29:43] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962583.769173}
  📡 [14:29:43] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962583.770035}
  📡 [14:29:43] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component

In [8]:
print('\n' + '=' * 60)
print('📋 Node 3: plan')
print('=' * 60)

from app.graph.nodes.plan import plan_node

state_after_plan = plan_node(state_after_thinking)

print(f'\n📊 结果:')
steps = state_after_plan.get('plan_steps', [])
print(f'   共 {len(steps)} 个步骤:')
for s in steps:
        print(f'   Step {s["id"]}: {s["label"]} (type={s.get("type", "?")})')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "plan" in e["type"]])}')


📋 Node 3: plan
  📡 [14:29:52] plan:start  {'id': 'plan_1777962592833', 'steps': [], 'current': 0, 'timestamp': 1777962592.8348958}


[Plan] LLM 输出 JSON 解析失败: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Expecting value: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Unterminated string starting at: line 1 column 1 (char 0)，使用降级方案
[Plan] LLM 输出 JSON 解析失败: Expecting value: line 1 column 1 (char 0)，使用降级方案
[stream_llm] LLM 调用失败 node=plan: 'int' object has no attribute 'get'
Traceback (most recent call last):
  File "/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend/app/graph/nodes/llm_utils.py", line 58, in stream_llm
    emit_delta(text)
    ~~~~~~~~~~^^^^^^
  File "/Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/proje

  📡 [14:30:00] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962600.398026}
  📡 [14:30:00] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962600.399594}
  📡 [14:30:00] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component'}, {'id': 3, 'label': '添加样式和交互', 'type': 'style'}, {'id': 4, 'label': '安装依赖并校验', 'type': 'deploy'}], 'is_complete': False, 'timestamp': 1777962600.400717}
  📡 [14:30:00] plan:update  {'steps': [{'id': 1, 'label': '初始化项目结构', 'type': 'init'}, {'id': 2, 'label': '生成核心组件', 'type': 'component

## Step 5: style_picker — 风格选择

根据 ui_style 选择风格配置

In [10]:
print('\n' + '=' * 60)
print('🎨 Node 4: style_picker')
print('=' * 60)

from app.graph.nodes.style_picker import style_picker_node

state_after_style = style_picker_node(state_after_plan)

print(f'\n📊 结果:')
print(f'   ui_style        = {state_after_style.get("ui_style")}')
print(f'   ui_style_config 长度 = {len(state_after_style.get("ui_style_config", ""))} chars')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "style" in e["type"]])}')

[StylePicker] CLI 不存在: /Users/fangyan/WorkBuddy/20260416190835/agent-projects-100/projects/pageforge/backend/app/skills/UI_UX/scripts/search.py，使用降级配置



🎨 Node 4: style_picker

📊 结果:
   ui_style        = minimal
   ui_style_config 长度 = 311 chars
   新增 SSE 事件数: 0


## Step 6: code_gen — 代码生成

生成项目文件 → files[]

In [11]:
print('\n' + '=' * 60)
print('⚡ Node 5: code_gen')
print('=' * 60)

from app.graph.nodes.code_gen import code_gen_node

state_after_code = code_gen_node(state_after_style)

print(f'\n📊 结果:')
files = state_after_code.get('files', [])
print(f'   生成 {len(files)} 个文件:')
for f in files:
        print(f'   📄 {f["path"]} ({f.get("language", "?")})')
print(f'   project_id   = {state_after_code.get("project_id")}')
print(f'   install_status = {state_after_code.get("install_status")}')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "file" in e["type"] or "tool" in e["type"]])}')


⚡ Node 5: code_gen
  📡 [14:34:11] file:created  {'file_path': 'package.json', 'name': 'package.json', 'language': 'json', 'size_bytes': None, 'timestamp': 1777962851.4554088}
  📡 [14:34:11] file:created  {'file_path': 'index.html', 'name': 'index.html', 'language': 'html', 'size_bytes': None, 'timestamp': 1777962851.4555562}
  📡 [14:34:11] file:created  {'file_path': 'vite.config.ts', 'name': 'vite.config.ts', 'language': 'typescript', 'size_bytes': None, 'timestamp': 1777962851.455592}
  📡 [14:34:11] file:created  {'file_path': 'tsconfig.json', 'name': 'tsconfig.json', 'language': 'json', 'size_bytes': None, 'timestamp': 1777962851.455623}
  📡 [14:34:11] tool_call:start  {'tool_id': 'code_gen_llm', 'name': 'generate_component', 'input': {'file': 'src/App.tsx'}, 'timestamp': 1777962851.4556491}


[CodeGen] App 组件生成失败: Request timed out.


  📡 [14:35:43] tool_call:end  {'tool_id': 'code_gen_llm', 'status': 'success', 'duration_ms': None, 'error': None, 'timestamp': 1777962943.195093}
  📡 [14:35:43] file:created  {'file_path': 'src/App.tsx', 'name': 'App.tsx', 'language': 'typescript', 'size_bytes': None, 'timestamp': 1777962943.195255}
  📡 [14:35:43] file:created  {'file_path': 'src/main.tsx', 'name': 'main.tsx', 'language': 'typescript', 'size_bytes': None, 'timestamp': 1777962943.19529}
  📡 [14:35:43] file:created  {'file_path': 'src/index.css', 'name': 'index.css', 'language': 'css', 'size_bytes': None, 'timestamp': 1777962943.1953301}
  📡 [14:35:43] status:generation_done  {'timestamp': 1777962943.19537}

📊 结果:
   生成 7 个文件:
   📄 package.json (json)
   📄 index.html (html)
   📄 vite.config.ts (typescript)
   📄 tsconfig.json (json)
   📄 src/App.tsx (typescript)
   📄 src/main.tsx (typescript)
   📄 src/index.css (css)
   project_id   = ff5a8e87
   install_status = success
   新增 SSE 事件数: 9


## Step 7: reply — 最终回复

LLM 生成总结回复 → response_message

In [12]:
print('\n' + '=' * 60)
print('💬 Node 6: reply')
print('=' * 60)

from app.graph.nodes.reply import reply_node

final_state = reply_node(state_after_code)

print(f'\n📊 结果:')
print(f'   response_message = {final_state.get("response_message", "")[:200]}')
print(f'   is_complete      = {final_state.get("is_complete")}')
print(f'   新增 SSE 事件数: {len([e for e in event_log if "text" in e["type"]])}')


💬 Node 6: reply
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '已成功', 'timestamp': 1777962955.010744}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '生成', 'timestamp': 1777962955.0113122}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '极简风格', 'timestamp': 1777962955.011743}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': ' Todo ', 'timestamp': 1777962955.012457}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '应用！', 'timestamp': 1777962955.012915}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '包含以下', 'timestamp': 1777962955.013927}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '功能', 'timestamp': 1777962955.014246}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '：\n\n', 'timestamp': 1777962955.014401}
  📡 [14:35:55] text:delta  {'id': 'text_1777962943210', 'content': '- ', 'timestamp': 1777962955.0145779}
  📡 [14:35:55] text:del

## 📊 全流程事件统计

汇总所有 SSE 事件

In [14]:
print('\n' + '=' * 60)
print('📊 全流程 SSE 事件统计')
print('=' * 60)

from collections import Counter

# 按类型分组
event_types = Counter(e['type'] for e in event_log)
print(f'\n共 {len(event_log)} 个事件:\n')

for etype, count in sorted(event_types.items()):
    print(f'  {etype:<25} x{count}')

print('\n--- 事件时间线 ---')
for e in event_log:
    data_preview = str(e['data'])[:80]
    print(f'  [{e["ts"]}] {e["type"]:<25} {data_preview}')


📊 全流程 SSE 事件统计

共 186 个事件:

  file:created              x7
  plan:done                 x3
  plan:start                x3
  plan:update               x20
  status:generation_done    x1
  text:delta                x30
  text:done                 x1
  thinking:delta            x117
  thinking:end              x1
  thinking:start            x1
  tool_call:end             x1
  tool_call:start           x1

--- 事件时间线 ---
  [14:26:32] thinking:start            {'id': 'thinking_1777962392234', 'timestamp': 1777962392.235397}
  [14:26:38] thinking:delta            {'id': 'thinking_1777962392234', 'delta': '1', 'timestamp': 1777962398.836864}
  [14:26:38] thinking:delta            {'id': 'thinking_1777962392234', 'delta': '.', 'timestamp': 1777962398.837996}
  [14:26:38] thinking:delta            {'id': 'thinking_1777962392234', 'delta': ' 需求', 'timestamp': 1777962398.838449}
  [14:26:38] thinking:delta            {'id': 'thinking_1777962392234', 'delta': '理解：', 'timestamp': 1777962398.838777}


## 🔀 场景2: chat 意图（走 shortcut 路径）

测试 `intent_router → reply` 的快捷路径

In [ ]:
print('\n' + '=' * 60)
print('🔀 场景2: chat 意图 (shortcut 路径)')
print('=' * 60)

# 清空事件日志
event_log.clear()

chat_state = {
    **initial_state,
    'user_message': '你好，PageForge 是什么？',
}

print(f'用户消息: {chat_state["user_message"]}')
print('\n--- Step 1: intent_router ---')

chat_after_intent = intent_router(chat_state)
print(f'intent = {chat_after_intent.get("intent")}')

# chat 意图 → 直接走 reply，跳过 thinking/plan/style_picker/code_gen
print('\n--- Step 2: reply (shortcut) ---')
chat_final = reply_node(chat_after_intent)
print(f'\nresponse_message = {chat_final.get("response_message", "")}')
print(f'is_complete = {chat_final.get("is_complete")}')
print(f'\n共 {len(event_log)} 个 SSE 事件')


🔀 场景2: chat 意图 (shortcut 路径)
用户消息: 你好，PageForge 是什么？

--- Step 1: intent_router ---
intent = chat

--- Step 2: reply (shortcut) ---
